In [1]:
print("hello")

hello


In [1]:
import os
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

In [ ]:

RAG_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL ="text-embedding-3-small"
DEEPEVAL_JUDGE_MODEL = "gpt-4.1-mini"

In [ ]:
os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "300"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "1"

In [ ]:
documents = [
    Document(
        page_content="Full-time employees receive 24 paid leaves per calendar year.",
        metadata={"doc_id": "leave_policy"},
    ),
    Document(
        page_content="Employees are allowed to work from home for a maximum of 2 days per week.",
        metadata={"doc_id": "remote_policy"},
    ),
    Document(
        page_content="Employees can claim up to ₹3000 per month for internet reimbursement.",
        metadata={"doc_id": "internet_policy"},
    ),
    Document(
        page_content="The standard probation period for new employees is 6 months.",
        metadata={"doc_id": "probation_policy"},
    ),
    Document(
        page_content="Employees receive ₹1000 per month as mobile reimbursement.",
        metadata={"doc_id": "mobile_policy"},
    ),
    Document(
        page_content="Medical insurance coverage begins from the employee's date of joining.",
        metadata={"doc_id": "insurance_policy"},
    ),
]

In [3]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

NameError: name 'OpenAIEmbeddings' is not defined

In [4]:

vector_store = InMemoryVectorStore(embedding=embeddings)

NameError: name 'InMemoryVectorStore' is not defined

In [5]:
vector_store.add_documents(documents)

NameError: name 'vector_store' is not defined

In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

NameError: name 'vector_store' is not defined

In [7]:

llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

NameError: name 'ChatOpenAI' is not defined

In [8]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY from the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not present in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }

In [9]:
sample = rag_pipeline("What is the monthly internet reimbursement limit?")

print("ANSWER:")
print(sample["answer"])

print("\nRETRIEVED DOCS:")
print(sample["retrieved_doc_ids"])

print("\nCONTEXT:")
for chunk in sample["retrieval_context"]:
    print("-", chunk)

NameError: name 'retriever' is not defined

In [10]:
goldens = [
    Golden(
        input="How many paid leaves does a full-time employee receive?",
        expected_output="A full-time employee receives 24 paid leaves per calendar year.",
    ),
    Golden(
        input="How many work-from-home days are allowed per week?",
        expected_output="Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="What is the monthly internet reimbursement limit?",
        expected_output="Employees can claim up to ₹3000 per month for internet reimbursement.",
    ),
    Golden(
        input="What is the probation period for new employees?",
        expected_output="The standard probation period for new employees is 6 months.",
    ),
    Golden(
        input="Can an employee work remotely for 3 days every week?",
        expected_output="No. Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="When does employee medical insurance coverage begin?",
        expected_output="Medical insurance coverage begins from the employee's date of joining.",
    ),
]

dataset = EvaluationDataset(goldens=goldens)

print("Goldens:", len(dataset.goldens))

NameError: name 'Golden' is not defined

In [11]:
rag_runs = []


dataset

NameError: name 'dataset' is not defined

In [12]:
dataset.goldens

NameError: name 'dataset' is not defined

In [13]:
for i, golden in enumerate(dataset.goldens, start=1):
    print(f"Running RAG {i}/{len(dataset.goldens)}")
    result = rag_pipeline(golden.input)
    rag_runs.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "actual_output": result["answer"],
        "retrieval_context": result["retrieval_context"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
    })

NameError: name 'dataset' is not defined

In [14]:
pd.DataFrame([
    {
        "input": run["input"],
        "expected_output": run["expected_output"],
        "actual_output": run["actual_output"],
        "retrieved_doc_ids": run["retrieved_doc_ids"],
    }
    for run in rag_runs
])

""


In [15]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [16]:
test_cases = [
    LLMTestCase(
        input=run["input"],
        actual_output=run["actual_output"],
        expected_output=run["expected_output"],
        retrieval_context=run["retrieval_context"],
    )
    for run in rag_runs
]

print("Test cases:", len(test_cases))

Test cases: 0


In [17]:
test_cases

[]

In [ ]:
DEEPEVAL_JUDGE_MODEL

In [18]:
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

faithfulness = FaithfulnessMetric(
    threshold=0.85,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_recall = ContextualRecallMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)
rag_metrics = [
    answer_relevancy,
    faithfulness,
    contextual_relevancy,
    contextual_precision,
    contextual_recall,
]

NameError: name 'DEEPEVAL_JUDGE_MODEL' is not defined

In [ ]:

first_test_case = test_cases[0]

In [ ]:
for metric in rag_metrics:
    print(metric)

In [ ]:
for metric in rag_metrics:
    print("\n", "=" * 70)
    print(metric.__class__.__name__)

In [19]:
for metric in rag_metrics:
    print("\n", "=" * 70)
    print(metric.__class__.__name__)
    metric.measure(first_test_case)
    print("Score:", metric.score)
    print("Passed:", metric.is_successful())
    print("Reason:", metric.reason)

NameError: name 'rag_metrics' is not defined

In [20]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

In [21]:
evaluation_result = evaluate(
    test_cases=test_cases[:1],
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

NameError: name 'rag_metrics' is not defined

In [22]:
evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

NameError: name 'rag_metrics' is not defined

In [ ]:
evaluation_result